# Retrain v3 — curated bridge/concrete mix

Trains **your custom U-Net** (SE blocks, bottleneck dropout, deep supervision) on UAV Kaggle + DeepCrack + Auto-ROS-LAB UAV 11k. Run cells top to bottom.

**Before you start (upload to MyDrive):**
- `crack-seg_revision_8-2.zip` — the repo code (get it from the project folder; if you've pushed `revision_8-2` to GitHub, you can skip this and the notebook will clone).
- `kaggle.json` — for the UAV dataset download (kaggle.com → Settings → API → Create New Token). Required unless you re-upload `dataset_split.zip` to `MyDrive/bridge_crack_detection/` to keep the original 220/47/48 split.
- DeepCrack's license is non-commercial research/educational — fine for this project.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/crack_v3'
DATA_ROOT = '/content/crack_data'
OUT = DATA_ROOT + '/dataset_split'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)
print('Data out :', OUT)

## 2. Get the code + dependencies

### (Optional) Where am I? — prints every path the notebook uses

In [ ]:
import os
print('CWD          :', os.getcwd())
print('MyDrive items:', sorted(os.listdir('/content/drive/MyDrive')))
print()
print('Notebook variables in this session:')
for v in ('REPO', 'DRIVE_ROOT', 'DATA_ROOT', 'OUT'):
    print(f'  {v:12s} = {globals().get(v)}')
print()
print('Expected paths:')
for label, p in [
    ('repo zip        ', '/content/drive/MyDrive/crack-seg_revision_8-2.zip'),
    ('repo folder     ', '/content/drive/MyDrive/crack-seg'),
    ('dataset_split   ', '/content/drive/MyDrive/bridge_crack_detection/dataset_split.zip'),
    ('kaggle.json     ', '/content/drive/MyDrive/kaggle.json'),
    ('DRIVE_ROOT (out)', '/content/drive/MyDrive/crack_v3'),
    ('DATA_ROOT       ', '/content/crack_data'),
    ('OUT             ', '/content/crack_data/dataset_split'),
]:
    print(f'  {label} {p}  ->  exists: {os.path.exists(p)}')

In [ ]:
import os, sys, shutil, glob

def find_repo():
    if os.path.isdir('/content/crack-seg'):
        return '/content/crack-seg'
    if os.path.isdir('/content/src') and os.path.isdir('/content/scripts'):
        return '/content'
    return None

REPO = find_repo()
if REPO is None:
    if os.path.isdir('/content/drive/MyDrive/crack-seg'):
        REPO = '/content/crack-seg'
        shutil.copytree('/content/drive/MyDrive/crack-seg', REPO)
        print('Copied repo from Drive folder.')
    else:
        zips = sorted(glob.glob('/content/drive/MyDrive/crack*seg*.zip'))
        zips += sorted(glob.glob('/content/drive/MyDrive/**/crack*seg*.zip', recursive=True))
        if zips:
            shutil.unpack_archive(zips[0], '/content')
            print('Unzipped repo from Drive:', zips[0])
        else:
            print('No repo on Drive yet. Pick crack-seg_revision_8-2.zip in the file dialog that just opened.')
            from google.colab import files
            uploaded = files.upload()
            for name in uploaded:
                if name.endswith('.zip'):
                    shutil.unpack_archive(name, '/content')
                    break
        REPO = find_repo() or '/content/crack-seg'
    if not os.path.isdir(REPO) and not (os.path.isdir('/content/src') and os.path.isdir('/content/scripts')):
        ret = os.system('git clone -b revision_8-2 https://github.com/Ishaan1402/crack-seg.git /content/crack-seg')
        if ret != 0 or not os.path.isdir('/content/crack-seg'):
            print('\nCould not get the repo code. Do one of:')
            print('  1) upload crack-seg_revision_8-2.zip to MyDrive and rerun this cell, or')
            print('  2) click the file dialog when this cell runs and pick the zip, or')
            print('  3) push revision_8-2 to GitHub and rerun this cell.')
            raise SystemExit(1)
        REPO = '/content/crack-seg'
os.chdir(REPO)
sys.path.insert(0, REPO)
# Minimal install ONLY: Colab already ships torch/numpy/opencv/pydantic.
# Installing requirements.txt would downgrade them and break the runtime.
os.system('pip install -q albumentations kagglehub gdown')
print('Setup complete in', os.getcwd())

### Kaggle credentials (required for the UAV source)

Provide them via **one** of: (1) pasting your username + API key into the cell below, (2) `MyDrive/kaggle.json`, or (3) Colab Secrets (`KAGGLE_USERNAME` + `KAGGLE_KEY`).

Find your credentials at kaggle.com → avatar → **Settings** → **API** → **Create New Token** (the current UI shows the username and key in-page — copy them).

In [ ]:
import json, os
from scripts import colab_data as cd

def _write_kaggle(user, key):
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump({'username': user, 'key': key}, f)
    os.environ['KAGGLE_USERNAME'] = str(user)
    os.environ['KAGGLE_KEY'] = str(key)

ok = cd.setup_kaggle_credentials('/content/drive/MyDrive/kaggle.json')
if ok:
    print('Kaggle credentials loaded from MyDrive/kaggle.json.')
else:
    try:
        from google.colab import userdata
        _write_kaggle(userdata.get('KAGGLE_USERNAME'), userdata.get('KAGGLE_KEY'))
        ok = True
        print('Kaggle credentials loaded from Colab Secrets.')
    except Exception:
        pass
if not ok:
    print('No kaggle.json in MyDrive and no Colab Secrets found.')
    print('Paste your username + API key into the next cell, or re-upload dataset_split.zip to')
    print('MyDrive/bridge_crack_detection/ to skip Kaggle entirely.')

In [ ]:
#@title Paste your Kaggle credentials here (kaggle.com -> Settings -> API)
KAGGLE_USERNAME = "your_kaggle_username" #@param {type:"string"}
KAGGLE_KEY = "paste_your_api_key_here" #@param {type:"string"}

import json, os
if KAGGLE_USERNAME.startswith('your_') or KAGGLE_KEY.startswith('paste_'):
    print('Replace the two placeholder values above with your real credentials, then rerun this cell.')
else:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY
    print('Kaggle credentials set from the form.')

## 3. Download & stage sources

### UAV Kaggle (primary — fresh 70/15/15 split via Kaggle, or original split if dataset_split.zip is on Drive)

In [ ]:
from scripts import colab_data as cd
DRIVE_ZIP = '/content/drive/MyDrive/bridge_crack_detection/dataset_split.zip'
mode = cd.download_uav(DRIVE_ZIP if os.path.exists(DRIVE_ZIP) else None, DATA_ROOT, OUT)
print('UAV staged from:', mode, '(drive = original 220/47/48 split, kaggle = fresh 70/15/15)')

### DeepCrack (train only; its test set is held out for evaluation)

In [ ]:
dc_train_img, dc_train_lab, DC_TEST_IMG, DC_TEST_LAB = cd.download_deepcrack(DATA_ROOT)
cd.stage(dc_train_img, dc_train_lab, OUT, source='deepcrack', cap=300, resize=512, val_frac=0.1, test_frac=0.0, seed=42)
print('DeepCrack test held out for eval:', DC_TEST_IMG)
print('                                 ', DC_TEST_LAB)

### Big volume source (Auto-ROS-LAB UAV 11k, with merged 11.2k fallback)

Tries the UAV 11k Drive file first (closest to your use case). When Google Drive rate-limits it, the cell automatically falls back to the merged 11.2k dataset (12 public crack datasets, 448×448, images+masks). Set `SKIP_BIG_SOURCE = True` to skip both and train on UAV Kaggle + DeepCrack only.

In [ ]:
import glob, os, gdown, zipfile
SKIP_BIG_SOURCE = False  # set True to skip both big sources

def _find_im_msk(root):
    def pick(kinds):
        cands = []
        for d in glob.glob(root + '/**/*', recursive=True):
            if os.path.isdir(d) and any(t in os.path.basename(d).lower() for t in kinds):
                n = len(os.listdir(d))
                if n > 10:
                    cands.append((n, d))
        top = [d for n, d in cands if os.path.dirname(d) == root]
        if top:
            return top[0]
        return sorted(cands, reverse=True)[0][1]
    return pick(('image', 'img')), pick(('mask', 'label', 'lab', 'gt'))

if not SKIP_BIG_SOURCE:
    staged = False
    try:
        imgs, msks = cd.download_uav11k(DATA_ROOT)
        cd.stage(imgs, msks, OUT, source='uav11k', cap=8000, resize=512, val_frac=0.1, test_frac=0.05, seed=42)
        staged = True
        print('UAV 11k staged (cap=8000).')
    except Exception as exc:
        print('UAV 11k failed (' + str(exc)[:100] + ')')
        print('Falling back to the merged 11.2k dataset...')
    if not staged:
        ZIP = DATA_ROOT + '/crack11k.zip'
        RAW = DATA_ROOT + '/crack11k_raw'
        try:
            if not os.path.exists(ZIP):
                gdown.download(id='1xrOqv0-3uMHjZyEUrerOYiYXW_E8SUMP', output=ZIP, quiet=False, fuzzy=True)
            if not (os.path.isdir(RAW) and any(glob.glob(RAW + '/**/*', recursive=True))):
                os.makedirs(RAW, exist_ok=True)
                with zipfile.ZipFile(ZIP) as z:
                    z.extractall(RAW)
            imgs, msks = _find_im_msk(RAW)
            print('Using images:', imgs, '| masks:', msks)
            cd.stage(imgs, msks, OUT, source='merged11k', cap=8000, resize=512, val_frac=0.1, test_frac=0.05, seed=42)
            print('Merged 11.2k staged (cap=8000).')
        except Exception as exc2:
            print('Merged 11.2k failed too:', exc2)
            print('Set SKIP_BIG_SOURCE = True to continue with UAV Kaggle + DeepCrack only.')

## 4. Staging summary (per-source sanity check)

In [ ]:
import json
with open(OUT + '/manifest.json') as f:
    manifest = json.load(f)
print(f"{'source':10s} {'train':>7s} {'val':>7s} {'test':>7s} {'crack%':>8s}")
for src, m in manifest.items():
    print(f"{src:10s} {m['train']:7d} {m['val']:7d} {m['test']:7d} {m['mean_crack_ratio']*100:7.2f}%")

## 5. Train v3 (your U-Net, upgraded)

- Primary run: narrow `[32,64,128,256]` + SE + dropout 0.1 + deep supervision, 512px, strong aug, AMP, lr 5e-4.
- If val Dice stays near 0 after a few epochs (the model predicts no cracks), stop and rerun with `--lr 1e-4` (and optionally `--bce-weight 0.7`).
- Optional wide comparison: set `RUN_WIDE = True` and rerun this cell.
- Hardware presets auto-select batch size (T4/L4 vs A100/V100).
- If the session dies, rerun with `--checkpoint` pointing at the last `.pth` to resume from those weights.

### Backup / restore staged data (use before a runtime factory reset)

If you ever need to factory-reset the runtime (e.g. after a broken pip install), run the backup cell to copy the downloaded + staged data to Drive, then after reset + Drive mount, run the restore line before the download cells so nothing is re-downloaded.

In [ ]:
import shutil
# BACKUP (run before factory reset):
shutil.copytree(DATA_ROOT, DRIVE_ROOT + '/crack_data_backup', dirs_exist_ok=True)
print('Backed up staged data to', DRIVE_ROOT + '/crack_data_backup')

# RESTORE (run after reset, once Drive is mounted and DATA_ROOT is set):
# shutil.copytree(DRIVE_ROOT + '/crack_data_backup', DATA_ROOT, dirs_exist_ok=True)
# print('Restored staged data from Drive.')

In [ ]:
from scripts import train as train_mod
RUN_WIDE = False
features = '64,128,256,512' if RUN_WIDE else '32,64,128,256'
out_name = 'unet_v3_wide.pth' if RUN_WIDE else 'unet_v3_narrow.pth'
out_path = DATA_ROOT + '/' + out_name
train_mod.main([
    '--data-dir', OUT,
    '--out', out_path,
    '--epochs', '30',
    '--lr', '5e-4',
    '--resize', '512',
    '--features', features,
    '--se', '--dropout', '0.1', '--deep-supervision',
    '--aug', 'strong',
    '--amp',
])

## 6. Save to Drive + evaluation commands

In [ ]:
import shutil
shutil.copy(out_path, DRIVE_ROOT + '/' + out_name)
print('Saved to Drive:', DRIVE_ROOT + '/' + out_name)
print()
print('HF upload (after huggingface-cli login):')
print(f'  huggingface-cli upload ishaan1402/crack-seg {out_path} unet_v3.pth')
print()
print('Cross-domain eval on DeepCrack test:')
print(f'  PYTHONPATH={REPO} python scripts/verify_metrics.py --checkpoint {out_path} --images {DC_TEST_IMG} --masks {DC_TEST_LAB} --mode both --thresholds 0.3 0.4 0.5 0.6 0.7')
print()
print('In-distribution eval on the staged test split (all sources, untouched during training):')
print(f'  PYTHONPATH={REPO} python scripts/verify_metrics.py --checkpoint {out_path} --images {OUT}/test/images --masks {OUT}/test/masks --mode both --thresholds 0.5')
print('  (files named uav_* in that split are the UAV Kaggle subset)')

## 7. Actual test run (same protocol for every model)

Runs the real evaluation now: the **staged test split** (untouched during training) and the **DeepCrack test** (never trained on). The v3 checkpoint from this session is compared against the two existing models fetched from `ishaan1402/crack-seg` on Hugging Face. Protocol: direct full-image inference, threshold 0.5, global + per-image Dice.

In [ ]:
import os
import numpy as np
import torch
import cv2
from src.models.checkpoint import load_unet_checkpoint

THRESH = 0.5
out_path = globals().get('out_path')
RUN_WIDE = globals().get('RUN_WIDE', False)

MODELS_DIR = DATA_ROOT + '/models'
os.makedirs(MODELS_DIR, exist_ok=True)
candidates = []

# 1) the v3 checkpoint trained in this session
if out_path and os.path.exists(out_path):
    candidates.append(('v3-' + ('wide' if RUN_WIDE else 'narrow'), out_path))

# 2) the two existing models from Hugging Face (public, no auth)
try:
    from huggingface_hub import hf_hub_download
    n2 = MODELS_DIR + '/unet_narrow_v2.pth'
    if not os.path.exists(n2):
        hf_hub_download(repo_id='ishaan1402/crack-seg', filename='best_unet.pth', local_dir=MODELS_DIR)
        os.rename(MODELS_DIR + '/best_unet.pth', n2)
    candidates.append(('narrow-v2 (published)', n2))
    w1 = MODELS_DIR + '/unet_wide_v1.pth'
    if not os.path.exists(w1):
        hf_hub_download(repo_id='ishaan1402/crack-seg', filename='unet_wide_v1.pth', local_dir=MODELS_DIR)
    if os.path.exists(w1):
        candidates.append(('wide-v1 (original)', w1))
except Exception as exc:
    print('Could not fetch existing models from HF:', exc)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict(model, rgb):
    x = torch.from_numpy(((rgb.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None]).to(device)
    with torch.inference_mode():
        return torch.sigmoid(model(x)).squeeze().cpu().numpy()

def run_set(name, img_dir, msk_dir):
    if not (os.path.isdir(img_dir) and os.path.isdir(msk_dir)):
        print(f'SKIP {name}: missing {img_dir} or {msk_dir}')
        return
    imgs, msks = sorted(os.listdir(img_dir)), sorted(os.listdir(msk_dir))
    print(f'\n=== {name} ({len(imgs)} images, threshold={THRESH}) ===')
    print(f"{'model':24s} {'Dice':>6s} {'IoU':>6s} {'Rec':>6s} {'Prec':>6s}")
    for label, ckpt in candidates:
        model, _ = load_unet_checkpoint(ckpt, device)
        g = {'tp': 0, 'fp': 0, 'fn': 0}
        per = []
        for ip, mp in zip(imgs, msks):
            rgb = cv2.cvtColor(cv2.imread(os.path.join(img_dir, ip)), cv2.COLOR_BGR2RGB)
            gt = cv2.imread(os.path.join(msk_dir, mp), cv2.IMREAD_GRAYSCALE) > 127
            pred = predict(model, rgb) > THRESH
            tp = int(np.sum(pred & gt)); fp = int(np.sum(pred & ~gt)); fn = int(np.sum(~pred & gt))
            g['tp'] += tp; g['fp'] += fp; g['fn'] += fn
            per.append((2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6))
        eps = 1e-6
        dice = (2 * g['tp'] + eps) / (2 * g['tp'] + g['fp'] + g['fn'] + eps)
        iou = (g['tp'] + eps) / (g['tp'] + g['fp'] + g['fn'] + eps)
        rec = (g['tp'] + eps) / (g['tp'] + g['fn'] + eps)
        prec = (g['tp'] + eps) / (g['tp'] + g['fp'] + eps)
        print(f"{label:24s} {dice:6.4f} {iou:6.4f} {rec:6.4f} {prec:6.4f}")
        print(f"{'':24s} macro Dice: {float(np.mean(per)):.4f}")

if not candidates:
    print('No checkpoints to evaluate (train first, or the HF download failed).')
else:
    run_set('staged test split', OUT + '/test/images', OUT + '/test/masks')
    dc_test = globals().get('DC_TEST_IMG')
    if dc_test:
        run_set('deepcrack test', dc_test, globals().get('DC_TEST_LAB'))

## Notes

- The staged dataset lives in Colab's ephemeral disk; only checkpoints are copied to Drive. Re-running the download cells after a session reset is expected.
- The UAV source now gets a fresh 70/15/15 split (your old dataset_split is gone). Treat that split as the fixed UAV test set going forward and do not regenerate it between runs.
- `manifest.json` records per-source counts + mean crack ratio — a source with a suspiciously low/high crack% is a red flag worth inspecting before trusting the run.
- After training, update the HF model card with test-set numbers only (run `verify_metrics.py` on the real UAV test split, not just validation).